# Nhập bộ keyframe + objects của BTC lên Modal Volume `keyframes-btc`

Kế hoạch: [`docs/triet/ke-hoach-offline-ingest.md`](../../docs/triet/ke-hoach-offline-ingest.md) §7, §7.1

**Bấm Run All. Bấm lại nếu phiên đứt** — video nào xong sẽ bị bỏ qua.

**Đây là lượt 1 của 3** trong thư mục `notebooks/btc/`: `02` nhập → `03` SigLIP+OCR → `04`
subtitle+CLAP. Cả ba cùng volume, cùng bundle, cùng cách chia người.

### Đây là bundle ĐỐI CHỨNG, không phải bundle chính

Volume **riêng** `keyframes-btc` (v2). Không dùng chung với bundle DAKE: ảnh và `objects/`
đều đánh địa chỉ theo số thứ tự `n`, nên trộn hai bộ là bundle không còn khớp
`map-keyframes` của chính nó (§1.2).

Làm bây giờ vì tham số DAKE còn phải hiệu chỉnh tiếp, còn bộ này **không phụ thuộc ρ hay
ngưỡng mật độ** — nó là chỗ dựa an toàn và là mốc so sánh cho §3.7. Nhưng nó **không phải bản
nháp**: keyframe của nó cố định, nên mọi thứ trích trên nó là kết quả cuối.

### Không GPU, không giải mã — chuyển dữ liệu thuần

| gói | dung lượng | nội dung |
|---|---|---|
| `Keyframes_*.zip` ×14 | **28,69 GB** | 177.321 ảnh, STORED, `keyframes/{video}/{n:03d}.jpg` |
| `objects-aic25-b1.zip` | 0,60 GB | 177.321 json, DEFLATE → 1,68 GB |
| `map-keyframes-aic25-b1.zip` | ~1 MB | 873 csv |
| `media-info-aic25-b1.zip` | ~1 MB | 873 json |

Số file `objects` khớp **chính xác** số keyframe — hai thứ dùng chung một `n`, và tầng nhập
từ chối nếu hai con số lệch nhau.

| người | GB | video | zip |
|---|---|---|---|
| triet | 5,67 | 88 | L25 |
| trong | 6,02 | 122 | L22, L23, L24, L29 |
| quang | 5,67 | 153 | L21, L26_c, L28 |
| bao | 5,76 | 295 | L26_a, L26_d, L30 |
| dat | 5,57 | 215 | L26_b, L26_e, L27 |

### Nếu tải chậm

Host `aic-data.ledo.io.vn` có bóp băng thông, và 5 người cùng kéo thì tranh nhau cùng một
đường. `wget` ở đây có `--read-timeout=60` nên kết nối chết sẽ tự nối lại, nhưng nếu tốc độ
**đều** ở mức vài KB/s thì đó là bị bóp, không phải đứt — restart session để đổi IP Kaggle,
và hẹn nhau chạy lệch giờ.

`-c` + kiểm kích thước nghĩa là chậm chỉ tốn thời gian, không hỏng dữ liệu.

### Cần có trước khi chạy

| | |
|---|---|
| Kaggle Secrets | `MODAL_TOKEN_ID`, `MODAL_TOKEN_SECRET`, `GITHUB_TOKEN` |
| Settings | Internet **On**. Accelerator: **bất kỳ** — tầng này không chạm GPU |

## Step 0a — repo + gói

In [ ]:
MEMBER      = "triet"          # <-- ĐỔI THÀNH TÊN BẠN
SHARDS      = "config/ingest_shards_batch1_btc.yaml"
TRANSPORT   = "modal"
VOLUME      = "keyframes-btc"  # volume RIÊNG, không phải aic2026
VOL_VERSION = 2
REPO_URL    = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF    = "feature/triet/structure"

import os, subprocess, sys
from pathlib import Path

REPO = Path("/kaggle/input/mfe-repo")
if not REPO.is_dir():
    REPO = Path("/kaggle/working/mfe")
    if not REPO.is_dir():
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{token}@")
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_REF, url, str(REPO)],
                       check=True, capture_output=True)   # token không lọt vào log
sys.path.insert(0, str(REPO / "src"))

# `nomodel` chứ không phải `keyframes`: tầng nhập không nạp model nào, và cùng
# file này được cả notebook DAKE dùng — nên tên nó không thuộc phương án nào.
# Các file requirements nằm ở gốc `notebooks/`, không nằm trong btc/ hay dake/.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO / "notebooks" / "requirements-kaggle-nomodel.txt")], check=True)
print("cài xong")

## Step 0b — Modal, phân công, việc đã xong

In [ ]:
import logging, yaml
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from kaggle_secrets import UserSecretsClient
from mfe.offline.bundle import BundleLayout, BundleWriter, StageStatus, load_shard_plan
from mfe.offline.stages import STAGES, StageContext
from mfe.offline.transport import build_transport

secrets = UserSecretsClient()
os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")

plan      = load_shard_plan(REPO / SHARDS)
layout    = BundleLayout(plan.bundle)
transport = build_transport(TRANSPORT, volume=VOLUME,
                            local_root="/kaggle/working/bundle_local",
                            version=VOL_VERSION)
WORK   = Path("/kaggle/working/scratch"); WORK.mkdir(parents=True, exist_ok=True)
ZIPS   = Path("/kaggle/working/zips");    ZIPS.mkdir(parents=True, exist_ok=True)
writer = BundleWriter(transport, layout, MEMBER, WORK)

assert plan.stages == ["n0_import_btc"], f"yaml khai {plan.stages}"
assert plan.keyframe_set == "btc"

EXTRAS    = yaml.safe_load((REPO / SHARDS).read_text(encoding="utf-8"))["extras"]
MY_ZIPS   = plan.zips_for(MEMBER)
MY_VIDEOS = plan.videos_for(MEMBER)
status    = StageStatus.load(transport, layout, MEMBER)
todo      = [v for v in MY_VIDEOS if not status.is_done("n0_import_btc", v)]

print(f"bundle  : {plan.bundle}  ->  volume {VOLUME} (v{VOL_VERSION})")
print(f"người   : {MEMBER}")
print(f"zip     : {', '.join(n for n, _ in MY_ZIPS)}")
print(f"video   : {len(MY_VIDEOS)} (tổng {len(plan.videos)})")
print(f"còn lại : {len(todo)}")

In [ ]:
import shutil, zipfile

def free_gb():
    return shutil.disk_usage("/kaggle/working").free / 2**30

def check_size(target, expect):
    """Kích thuoc dung hay khong. `expect=None` = khong biet, bo qua.

    Vi sao doc rieng chu khong tin `wget` bao thanh cong: `-c` noi lai bang HTTP
    Range, va **neu may chu lo Range** thi no tra ve ca file voi ma 200, con
    wget noi phan do vao sau phan da co. File ra to hon that, hong tu giua, va
    khong co canh bao nao. Kiem kich thuoc la thu duy nhat thay duoc dieu do.
    Cat cut (dut giua chung) thi cung bat duoc luon.
    """
    if expect is None:
        return
    actual = target.stat().st_size
    if actual != expect:
        target.unlink(missing_ok=True)
        raise RuntimeError(
            f"{target.name}: tai ve {actual:,} byte, cho doi {expect:,} "
            f"(lech {actual - expect:+,}). Da xoa file hong — chay lai o nay."
        )


def download(url, name, expect_bytes=None):
    """Tải một gói, có resume. `wget -c` chạy tiếp từ chỗ đứt."""
    target = ZIPS / name
    if target.is_file():
        size = target.stat().st_size
        if expect_bytes is None or size == expect_bytes:
            print(f"    {name}: đã có ({size / 2**30:.2f} GB)")
            return target
        # File dang do tu phien truoc — `wget -c` chay tiep, khong tai lai.
        print(f"    {name}: đang dở {size / 2**30:.2f}/{expect_bytes / 2**30:.2f} GB, chạy tiếp")
    print(f"    tải {name} … (đĩa trống {free_gb():.1f} GB)")
    # --read-timeout la thu QUAN TRONG nhat o day. Khong co no, mot ket noi bi
    # bop xuong ~400 B/s van la ket noi "song": wget ngoi doi vo han thay vi noi
    # lai. Co no thi 60 giay khong nhan duoc byte nao la ngat va thu lai, va lan
    # noi lai thuong duoc toc do binh thuong.
    #   --tries=0        thu lai mai (van an toan: -c khong tai lai tu dau)
    #   --waitretry=10   khong dap lien tuc vao host dang qua tai
    subprocess.run(
        ["wget", "-q", "--show-progress", "-c",
         "--timeout=30", "--read-timeout=60", "--tries=0", "--waitretry=10",
         "--retry-connrefused", "-O", str(target), url],
        check=True,
    )
    check_size(target, expect_bytes)
    return target

def extract_video(zip_path, video_id, into):
    """Bung các file của ĐÚNG một video ra `into/{video_id}/`.

    Bung từng video thay vì cả gói: gói keyframe lớn nhất là 2,4 GB nên bung cả
    gói vẫn vừa đĩa, nhưng làm theo video thì đỉnh dùng đĩa không phụ thuộc gói,
    và mỗi video xong là xoá ngay — cùng một cách cho mọi gói.
    """
    out = into / video_id
    out.mkdir(parents=True, exist_ok=True)
    count = 0
    with zipfile.ZipFile(zip_path) as archive:
        for name in archive.namelist():
            parts = Path(name).parts
            if len(parts) < 2 or parts[-2] != video_id or name.endswith("/"):
                continue
            with archive.open(name) as src, (out / parts[-1]).open("wb") as dst:
                shutil.copyfileobj(src, dst, length=1 << 20)
            count += 1
    return out, count

def extract_one_file(zip_path, inner_suffix, dest):
    with zipfile.ZipFile(zip_path) as archive:
        match = [n for n in archive.namelist() if n.endswith(inner_suffix)]
        if not match:
            raise FileNotFoundError(f"{inner_suffix} khong co trong {zip_path.name}")
        dest.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(match[0]) as src, dest.open("wb") as dst:
            shutil.copyfileobj(src, dst)
    return dest

# Ba gói nhỏ, dùng chung cho mọi video — tải một lần, giữ suốt phiên.
print("gói dùng chung:")
OBJECTS_ZIP = download(EXTRAS["objects"], "objects-aic25-b1.zip")
MAPS_ZIP    = download(EXTRAS["map_keyframes"], "map-keyframes-aic25-b1.zip")
print(f"đĩa trống: {free_gb():.1f} GB")

## Step 1 — vòng lặp

Mỗi gói keyframe: tải (có resume) → với từng video: bung ảnh + objects + `map-keyframes` →
đẩy lên Volume → **rồi mới cắm mốc** → xoá sạch. Xong gói thì xoá file zip.

Tầng nhập từ chối nếu thiếu ảnh hoặc nếu số file `objects` khác số ảnh — hai thứ đó đánh địa
chỉ theo cùng một `n`, và lệch nhau là bundle tự mâu thuẫn.

In [ ]:
import time, traceback

failures = []
started = time.perf_counter()
done_count = 0

for zip_name, zip_url in MY_ZIPS:
    pending = [v for v in plan.videos_in(zip_name)
               if not status.is_done("n0_import_btc", v)]
    if not pending:
        print()
        print(f"=== {zip_name}: xong hết, không tải ===")
        continue

    print()
    print(f"=== {zip_name}: còn {len(pending)}/{len(plan.videos_in(zip_name))} video ===")
    kf_zip = download(zip_url, zip_name, plan.zip_bytes.get(zip_name))

    for video_id in pending:
        try:
            kf_dir, n_img = extract_video(kf_zip, video_id, WORK / "keyframes")
            obj_dir, n_obj = extract_video(OBJECTS_ZIP, video_id, WORK / "objects")
            map_csv = extract_one_file(MAPS_ZIP, f"/{video_id}.csv",
                                       WORK / "map-keyframes" / f"{video_id}.csv")
            ctx = StageContext(video_id, Path("/dev/null"), [], writer, layout, WORK,
                               params={"keyframes_dir": str(kf_dir),
                                       "objects_dir": str(obj_dir),
                                       "map_csv": str(map_csv)})
            marker = STAGES["n0_import_btc"](ctx)
            status.done.add(("n0_import_btc", video_id))
            # merge=True: bundle nay do BA notebook cung do (02/03/04). Ghi lai manifest
            # chi tu marker cua luot nay se XOA artifact cua luot khac.
            writer.write_part_manifest(video_id, [marker], keyframes=marker["keyframes"], merge=True)
            done_count += 1
            rate = (time.perf_counter() - started) / done_count
            left = (len(todo) - done_count) * rate / 3600
            print(f"[{done_count:>3}/{len(todo)}] {video_id}  {marker['keyframes']:>4} kf  "
                  f"{marker['objects']:>4} obj  {marker['bytes'] / 2**20:>6.0f} MB  "
                  f"{marker['elapsed_s']}s | còn ~{left:.1f}h")
        except Exception as err:
            failures.append((video_id, repr(err)))
            print(f"[LỖI] {video_id}: {err}")
            traceback.print_exc(limit=3)
        finally:
            shutil.rmtree(WORK / "keyframes" / video_id, ignore_errors=True)
            shutil.rmtree(WORK / "objects" / video_id, ignore_errors=True)
            (WORK / "map-keyframes" / f"{video_id}.csv").unlink(missing_ok=True)

    kf_zip.unlink(missing_ok=True)
    print(f"    đã xoá {zip_name}, đĩa trống {free_gb():.1f} GB")

print()
print("=" * 72)
print(f"phiên này: {done_count} video xong, {len(failures)} lỗi, "
      f"{(time.perf_counter() - started) / 3600:.1f}h")
for video_id, err in failures:
    print(f"  {video_id}: {err}")

## Step 2 — chốt phiên

In [ ]:
fresh = StageStatus.load(transport, layout, MEMBER)
left  = [v for v in MY_VIDEOS if not fresh.is_done("n0_import_btc", v)]

print(f"{MEMBER}: {len(MY_VIDEOS) - len(left)}/{len(MY_VIDEOS)} video đã lên {VOLUME}")
if left:
    print(f"còn {len(left)} — bấm Run All lại:")
    print("  " + ", ".join(left[:12]) + (" …" if len(left) > 12 else ""))
else:
    print("XONG phần của bạn. Báo Lead.")
    print(f"  python scripts/verify_bundle.py --shards {SHARDS} --member {MEMBER}")